In [1]:
## Database Setup

import json
import sqlite3
from pathlib import Path

VALIDATED_DIR = Path("data/processed/validated")
DB_PATH = Path("data/processed/invoices.db")

def get_connection():
    return sqlite3.connect(DB_PATH)

In [2]:
## Created a table

def create_table():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS invoices (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            filename TEXT UNIQUE,
            vendor TEXT,
            invoice_number TEXT,
            invoice_date TEXT,
            net_amount REAL,
            vat_rate REAL,
            vat_amount REAL,
            total_amount REAL,
            currency TEXT,
            status TEXT,
            errors TEXT,
            warnings TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.commit()
    conn.close()
    print("Table 'invoices' ready.")


create_table()

Table 'invoices' ready.


In [3]:
## Loaded validated JSON and inserted into DB

def insert_invoice(record, filename):
    conn = get_connection()
    cursor = conn.cursor()

    data = record["data"]

    cursor.execute("""
        INSERT OR REPLACE INTO invoices (
            filename, vendor, invoice_number, invoice_date,
            net_amount, vat_rate, vat_amount, total_amount,
            currency, status, errors, warnings
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        filename,
        data.get("vendor"),
        data.get("invoice_number"),
        data.get("invoice_date"),
        data.get("net_amount"),
        data.get("vat_rate"),
        data.get("vat_amount"),
        data.get("total_amount"),
        data.get("currency"),
        record["status"],
        json.dumps(record["errors"], ensure_ascii=False),
        json.dumps(record["warnings"], ensure_ascii=False),
    ))

    conn.commit()
    conn.close()


def store_all_validated_invoices():
    validated_files = list(VALIDATED_DIR.glob("*.json"))

    if not validated_files:
        print("No validated files found in", VALIDATED_DIR)
        return

    inserted = 0
    for file in validated_files:
        record = json.loads(file.read_text(encoding="utf-8"))
        insert_invoice(record, file.name)
        inserted += 1
        print(f"Stored: {file.name} [{record['status']}]")

    print(f"\n{inserted} invoice(s) stored in {DB_PATH}")


store_all_validated_invoices()

Stored: invoice_001.json [PASSED]
Stored: invoice_003.json [PASSED]
Stored: invoice_002.json [PASSED]

3 invoice(s) stored in data/processed/invoices.db


In [4]:
## Confirming if it's working or not

import pandas as pd

def view_all_invoices():
    conn = get_connection()
    df = pd.read_sql_query("SELECT * FROM invoices", conn)
    conn.close()
    return df


df = view_all_invoices()
df

,id,filename,vendor,invoice_number,invoice_date,net_amount,vat_rate,vat_amount,total_amount,currency,status,errors,warnings,created_at
0,1,invoice_001.json,Buerobedarf Mueller GmbH,RE-2026-0451,2026-07-05,250.0,19.0,47.50,297.50,EUR,PASSED,[],[],2026-07-16 17:33:48
1,2,invoice_003.json,Bio-Catering Berlin,BC-2026-077,2026-07-12,85.5,7.0,5.99,91.49,EUR,PASSED,[],[],2026-07-16 17:33:48
2,3,invoice_002.json,Software Solutions AG,INV-9981,2026-07-10,1200.0,19.0,228.00,1428.00,EUR,PASSED,[],[],2026-07-16 17:33:48


In [5]:
## Some useful queries

conn = get_connection()

# Total VAT collected across all invoices
total_vat = pd.read_sql_query("SELECT SUM(vat_amount) as total_vat FROM invoices", conn)
print("Total VAT:", total_vat["total_vat"][0])

# Invoices flagged with warnings or failures
flagged = pd.read_sql_query(
    "SELECT filename, vendor, status, errors, warnings FROM invoices WHERE status != 'PASSED'",
    conn
)
print("\nFlagged invoices:")
print(flagged)

conn.close()

Total VAT: 281.49

Flagged invoices:
Empty DataFrame
Columns: [filename, vendor, status, errors, warnings]
Index: []


In [ ]:
## Data Storage with SQLite (`04_store_data.ipynb`)

### What this notebook does
This is the storage stage of the pipeline. It takes the validated invoice records from Step 3 and loads them into a SQLite database, turning scattered JSON files into a single, queryable source of truth.

### Process
1. **Create the database schema** — a SQLite table (`invoices`) is created with columns for every extracted field (vendor, invoice number, date, net/VAT/total amounts, currency), plus the validation status and any errors/warnings from Step 3. Each record also gets an auto-incrementing ID and a timestamp.
2. **Load and insert validated records** — every validated JSON file is read and inserted into the database. `INSERT OR REPLACE` is used so re-running the notebook updates existing records instead of creating duplicates, making the pipeline safely re-runnable.
3. **Verify with queries** — the notebook loads the full table into a pandas DataFrame to visually confirm the data looks correct, and runs a couple of example SQL queries (total VAT collected, invoices flagged with issues) to demonstrate the database is actually useful, not just a dump.

### Why SQLite (and why a database at all)
Keeping validated invoices in separate JSON files works for a handful of test invoices, but doesn't scale — you can't easily filter, sum, or report on data spread across many files. A database turns the pipeline's output into something a real business could actually use: run VAT totals for tax reporting, filter by vendor, or pull up all invoices that need manual review. SQLite specifically was chosen because it requires no server setup, making the project fully self-contained and easy for anyone to run locally — appropriate for a small-business tool.

### Design choices worth noting
- **Validation status is stored alongside the data**, not discarded — so flagged invoices remain visible and queryable rather than silently dropped.
- **`INSERT OR REPLACE` on filename** makes the notebook idempotent — safe to re-run without creating duplicate entries.
- **Errors/warnings stored as JSON strings** in their own columns, preserving the full validation context per invoice for auditing.

### Output
- `data/processed/invoices.db` — a SQLite database containing all processed invoices, ready to be queried or connected to a dashboard.

### Next step
`05_dashboard.ipynb` or `app.py` — build a Streamlit interface to upload new invoices, view stored data, filter/search, and export results to Excel/CSV.